In [1]:
import pandas as pd

# Load the original IBM Telco Customer Churn dataset
file_path = "data/Telco-Customer-Churn.csv"
df = pd.read_csv(file_path)

In [2]:

# Check missing values in each column
missing_values = df.isnull().sum()

print("Missing values in each column:")
print(missing_values)

# Check total missing values in the dataset
print("\nTotal missing values:", missing_values.sum())


Missing values in each column:
customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

Total missing values: 0


In [3]:

# Check for empty or whitespace-only strings in each column
blank_values = df.select_dtypes(include="object").apply(
    lambda col: col.str.fullmatch(r"\s*").fillna(False).sum()
)

print("Blank or whitespace-only values:")
print(blank_values[blank_values > 0])


Blank or whitespace-only values:
TotalCharges    11
dtype: int64


In [4]:
# Verifying relationship between tenure and charges
import numpy as np

# Convert TotalCharges to numeric for comparison
# Blank values will become NaN
total_charges = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Calculate expected TotalCharges
expected_total = df["MonthlyCharges"] * df["tenure"]

# Compare actual and expected TotalCharges
# Ignore rows where TotalCharges is missing
valid_rows = total_charges.notna()

matches = np.isclose(
    total_charges[valid_rows],
    expected_total[valid_rows],
    atol=0.01,
    rtol=0
)

print("Total records:", len(df))
print("Records with valid TotalCharges:", valid_rows.sum())
print("Records where formula matches:", matches.sum())
print("Records where formula differs:", (~matches).sum())

print(
    "Percentage matching:",
    round(matches.mean() * 100, 2), "%"
)


Total records: 7043
Records with valid TotalCharges: 7032
Records where formula matches: 614
Records where formula differs: 6418
Percentage matching: 8.73 %


In [5]:
# examining the differences

# Create a temporary comparison DataFrame
comparison = df[
    ["customerID", "tenure", "MonthlyCharges"]
].copy()

comparison["ActualTotalCharges"] = total_charges
comparison["ExpectedTotalCharges"] = expected_total

comparison["Difference"] = (
    comparison["ActualTotalCharges"]
    - comparison["ExpectedTotalCharges"]
)

# Display records where the difference exceeds $0.01
different_records = comparison[
    comparison["ActualTotalCharges"].notna()
    & (comparison["Difference"].abs() > 0.01)
]

display(different_records.head(20))


,customerID,tenure,MonthlyCharges,ActualTotalCharges,ExpectedTotalCharges,Difference
1,5575-GNVDE,34,56.95,1889.50,1936.30,-46.80
2,3668-QPYBK,2,53.85,108.15,107.70,0.45
3,7795-CFOCW,45,42.30,1840.75,1903.50,-62.75
4,9237-HQITU,2,70.70,151.65,141.40,10.25
5,9305-CDSKC,8,99.65,820.50,797.20,23.30
6,1452-KIOVK,22,89.10,1949.40,1960.20,-10.80
7,6713-OKOMC,10,29.75,301.90,297.50,4.40
8,7892-POOKP,28,104.80,3046.05,2934.40,111.65
9,6388-TABGU,62,56.15,3487.95,3481.30,6.65
10,9763-GRSKD,13,49.95,587.45,649.35,-61.90


In [6]:
# specifically verifying the customers with tenure = 1

tenure_1 = comparison[comparison["tenure"] == 1].copy()

tenure_1["ChargesMatch"] = np.isclose(
    tenure_1["ActualTotalCharges"],
    tenure_1["ExpectedTotalCharges"],
    atol=0.01,
    rtol=0
)

print("Customers with tenure = 1:", len(tenure_1))
print("Matching charges:", tenure_1["ChargesMatch"].sum())
print("Non-matching charges:", (~tenure_1["ChargesMatch"]).sum())

display(tenure_1.head(10))


Customers with tenure = 1: 613
Matching charges: 613
Non-matching charges: 0


,customerID,tenure,MonthlyCharges,ActualTotalCharges,ExpectedTotalCharges,Difference,ChargesMatch
0,7590-VHVEG,1,29.85,29.85,29.85,0.0,True
20,8779-QRDMV,1,39.65,39.65,39.65,0.0,True
22,1066-JKSGK,1,20.15,20.15,20.15,0.0,True
27,8665-UTDHZ,1,30.20,30.20,30.20,0.0,True
33,7310-EGVHZ,1,20.20,20.20,20.20,0.0,True
34,3413-BMNZE,1,45.25,45.25,45.25,0.0,True
70,2273-QCKXA,1,49.05,49.05,49.05,0.0,True
80,5919-TMRGD,1,79.35,79.35,79.35,0.0,True
91,2424-WVHPL,1,74.70,74.70,74.70,0.0,True
100,6380-ARCEH,1,20.20,20.20,20.20,0.0,True


In [7]:
# frequency distribution of categorical columns

# List of qualitative (categorical) columns
categorical_columns = [
    "gender",
    "SeniorCitizen",
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod",
    "Churn"
]

# Display the frequency distribution for each column
for col in categorical_columns:
    print(f"\n{'=' * 50}")
    print(f"Column: {col}")
    print(f"{'=' * 50}")

    frequency = df[col].value_counts(dropna=False)

    print(frequency)



Column: gender
gender
Male      3555
Female    3488
Name: count, dtype: int64

Column: SeniorCitizen
SeniorCitizen
0    5901
1    1142
Name: count, dtype: int64

Column: Partner
Partner
No     3641
Yes    3402
Name: count, dtype: int64

Column: Dependents
Dependents
No     4933
Yes    2110
Name: count, dtype: int64

Column: PhoneService
PhoneService
Yes    6361
No      682
Name: count, dtype: int64

Column: MultipleLines
MultipleLines
No                  3390
Yes                 2971
No phone service     682
Name: count, dtype: int64

Column: InternetService
InternetService
Fiber optic    3096
DSL            2421
No             1526
Name: count, dtype: int64

Column: OnlineSecurity
OnlineSecurity
No                     3498
Yes                    2019
No internet service    1526
Name: count, dtype: int64

Column: OnlineBackup
OnlineBackup
No                     3088
Yes                    2429
No internet service    1526
Name: count, dtype: int64

Column: DeviceProtection
DeviceProtec

In [8]:
# Verify InternetService and its six dependent services

# Internet-related service columns
internet_features = [
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

# Customers without internet service
no_internet = df["InternetService"] == "No"

# Check whether all six columns say "No internet service"
all_no_internet = df[internet_features].eq(
    "No internet service"
).all(axis=1)

# Identify records that violate the expected relationship
internet_mismatches = df[
    no_internet != all_no_internet
]

print("Customers without internet:", no_internet.sum())
print("Customers with all six services marked 'No internet service':",
      all_no_internet.sum())
print("Records violating the relationship:", len(internet_mismatches))

# Display exceptions, if any
display(
    internet_mismatches[
        ["customerID", "InternetService"] + internet_features
    ]
)


Customers without internet: 1526
Customers with all six services marked 'No internet service': 1526
Records violating the relationship: 0


,customerID,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies


In [9]:
# verifying phone service and multiple lines.

# Customers without phone service
no_phone = df["PhoneService"] == "No"

# Customers with MultipleLines marked "No phone service"
no_phone_lines = df["MultipleLines"] == "No phone service"

# Identify records violating the expected relationship
phone_mismatches = df[
    no_phone != no_phone_lines
]

print("Customers without phone service:", no_phone.sum())
print("Customers with MultipleLines = 'No phone service':",
      no_phone_lines.sum())
print("Records violating the relationship:", len(phone_mismatches))

# Display exceptions, if any
display(
    phone_mismatches[
        ["customerID", "PhoneService", "MultipleLines"]
    ]
)


Customers without phone service: 682
Customers with MultipleLines = 'No phone service': 682
Records violating the relationship: 0


,customerID,PhoneService,MultipleLines


In [10]:
# check for duplicate records and customerIDs
print("Duplicate rows:", df.duplicated().sum())

print(
    "Duplicate customer IDs:",
    df["customerID"].duplicated().sum()
)

Duplicate rows: 0
Duplicate customer IDs: 0


In [11]:
# validating column datatype
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [12]:
# check invalid numerical values
# Check for negative values
print("Negative tenure:",
      (df["tenure"] < 0).sum())

print("Negative MonthlyCharges:",
      (df["MonthlyCharges"] < 0).sum())

# Check tenure range
print("Minimum tenure:", df["tenure"].min())
print("Maximum tenure:", df["tenure"].max())

# Check MonthlyCharges range
print("Minimum MonthlyCharges:", df["MonthlyCharges"].min())
print("Maximum MonthlyCharges:", df["MonthlyCharges"].max())

# Check TotalCharges without modifying the dataset
total_numeric = pd.to_numeric(
    df["TotalCharges"], errors="coerce"
)

print("Negative TotalCharges:",
      (total_numeric < 0).sum())

Negative tenure: 0
Negative MonthlyCharges: 0
Minimum tenure: 0
Maximum tenure: 72
Minimum MonthlyCharges: 18.25
Maximum MonthlyCharges: 118.75
Negative TotalCharges: 0


In [13]:
# checking for logical inconsistency
# Customers with neither phone nor internet service
no_core_service = df[
    (df["PhoneService"] == "No") &
    (df["InternetService"] == "No")
]

print(
    "Customers without phone or internet service:",
    len(no_core_service)
)

display(no_core_service)

Customers without phone or internet service: 0


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn


In [14]:

# Convert TotalCharges to numeric
# Preserve the 11 blank values as NaN
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

# Verify the data type and missing values
print(df["TotalCharges"].dtype)
print(df["TotalCharges"].isna().sum())


float64
11
